In [ ]:
# Notebook: Interactive Complex Integration & Time-Domain Response (Slider for alpha)
# Function to invert: X(j\omega) = 1 / (\alpha + j\omega)

import sympy as sp
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider
from IPython.display import clear_output, display

# Define symbols
t, omega = sp.symbols('t omega', real=True)
alpha_sym = sp.symbols('alpha', real=True)

# 1. Symbolic Residue Calculation
X_omega = 1 / (alpha_sym + sp.I * omega)
integrand = X_omega * sp.exp(sp.I * omega * t)
pole = -alpha_sym / sp.I  # i.e., j * alpha_sym

print("--- 1. SYMBOLIC RESIDUE & INVERSE TRANSFORM ---")
residue_val = sp.residue(integrand, omega, pole)
x_t_result = sp.simplify((1 / (2 * sp.pi)) * (2 * sp.pi * sp.I * residue_val))
print("Derived inverse Fourier transform x(t) for t > 0:")
display(x_t_result)


# ==========================================================
# 2. INTERACTIVE VISUALIZATION (Complex Plane & Time Domain)
# ==========================================================
def plot_interactive_complex_system(alpha=1.5):
    # Clear previous output for smooth slider updates
    clear_output(wait=True)
    
    # Create a figure with two subplots (Top: Complex Plane, Bottom: Time Response)
    fig, (ax_complex, ax_time) = plt.subplots(2, 1, figsize=(9, 9))
    
    # ------------------------------------------------------
    # SUBPLOT 1: Complex Plane & Contour (Top)
    # ------------------------------------------------------
    ax_complex.axhline(0, color='black', linewidth=1)
    ax_complex.axvline(0, color='black', linewidth=1)
    
    R = 4.0
    ax_complex.plot([-R, R], [0, 0], 'r-', linewidth=2, label=r"Real Axis Segment $C_0$")
    
    # Semicircle path (Upper if alpha >= 0, Lower if alpha < 0 to enclose the pole properly)
    theta = np.linspace(0, np.pi, 200) if alpha >= 0 else np.linspace(np.pi, 2 * np.pi, 200)
    omega_R_real = R * np.cos(theta)
    omega_R_imag = R * np.sin(theta)
    path_label = r"Upper Semicircle $C_R$ ($R \to \infty$)" if alpha >= 0 else r"Lower Semicircle $C_R$ ($R \to \infty$)"
    ax_complex.plot(omega_R_real, omega_R_imag, 'g--', linewidth=2, label=path_label)
    
    # Plot the pole at \omega = j*alpha
    pole_imag = alpha
    ax_complex.plot(0, pole_imag, 'bx', markersize=12, markeredgewidth=3, label=rf"Pole at $\omega = j({alpha:.1f})$")
    
    ax_complex.set_title(r"Complex Integration Contour and Pole Location in the $\omega$-Plane", fontsize=11)
    ax_complex.set_xlabel(r"Re($\omega$)", fontsize=10)
    ax_complex.set_ylabel(r"Im($\omega$)", fontsize=10)
    ax_complex.grid(True, linestyle=":", alpha=0.7)
    ax_complex.set_xlim(-5, 5)
    ax_complex.set_ylim(-5, 5)
    
    # Fix aspect ratio so the circle looks like a true circle
    ax_complex.set_aspect('equal')
    
    # Move legend OUTSIDE the plot to the top right
    ax_complex.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=9)
    
    # ------------------------------------------------------
    # SUBPLOT 2: Time-Domain Response x(t) = e^{-\alpha t} (Bottom)
    # ------------------------------------------------------
    t_vals = np.linspace(0, 5, 500)
    with np.errstate(over='ignore', invalid='ignore'):
        x_vals = np.exp(-alpha * t_vals)
        
    ax_time.plot(t_vals, x_vals, 'b-', linewidth=2.5, label=r"$x(t) = e^{-(" + f"{alpha:.1f}" + r")t} u(t)$")
    
    ax_time.set_title(r"Corresponding Time-Domain Response $x(t)$", fontsize=11)
    ax_time.set_xlabel(r"Time $t$ (s)", fontsize=10)
    ax_time.set_ylabel(r"$x(t)$", fontsize=10)
    ax_time.grid(True, linestyle=":", alpha=0.7)
    ax_time.axhline(0, color='black', linewidth=0.8, linestyle='--')
    
    # Dynamically compute y-limits based on extreme alpha values in the slider [-2.0, 2.0]
    # For alpha = -2.0 at t = 5, e^(2*5) = e^10 (~22000), so we adapt the ylim dynamically
    max_val = np.nanmax(x_vals)
    min_val = np.nanmin(x_vals)
    upper_limit = min(max(max_val * 1.1, 2.0), 50.0)  # cap to prevent infinite blowout for extreme negative views
    lower_limit = max(min_val - 0.5, -2.0)
    
    ax_time.set_xlim(0, 5)
    ax_time.set_ylim(lower_limit, upper_limit)
    
    # Move legend outside for consistency
    ax_time.legend(loc='upper left', bbox_to_anchor=(1.02, 1), fontsize=10)
    
    plt.tight_layout()
    plt.show()
    plt.close(fig)

# Execute interactive widget with slider for alpha (ranging from -2.0 to 2.0)
interact(
    plot_interactive_complex_system, 
    alpha=FloatSlider(value=1.5, min=-2.0, max=2.0, step=0.1, description=r"Parameter $\alpha$", style={'description_width': 'initial'})
);